# Bielik - agent pogodowy z surowym SDK OpenAI

[Bielik](https://bielik.ai/) to polski model językowy stworzony przez [SpeakLeash](https://speakleash.org/) i ICM. W tym notebooku zbudujemy agenta pogodowego (analogicznego do tego z `003-01. LLM-function_calling.ipynb`) opartego o model `SpeakLeash/bielik-11b-v3.0-instruct:bf16` hostowany lokalnie z użyciem [Ollama](https://ollama.com/).

Pokazujemy tu **najniższą warstwę** - manualną pętlę function callingu na surowym SDK [openai](https://github.com/openai/openai-python), bez frameworka agentowego. Pozostałe trzy notebooki w tej serii pokazują tę samą funkcjonalność opakowaną przez różne frameworki:
- `010-02. Bielik-pydantic-ai.ipynb`
- `010-03. Bielik-dspy.ipynb`
- `010-04. Bielik-llama-index.ipynb`

## Dlaczego potrzebny jest custom Modelfile?

Bielik (jak wiele modeli trenowanych w stylu _Hermes function calling_) emituje wywołania funkcji w polu `content` jako tagi `<tool_call>{...}</tool_call>`. Standardowo Ollama nie konwertuje ich na ustrukturyzowane pole `tool_calls` w odpowiedzi - aplikacja musiałaby je sama parsować.

Rozwiązanie: nadpisujemy template Bielika własnym Modelfile (`010-00. Bielik.Modelfile`), który deklaruje tagi `<tool_call>...</tool_call>` przylegle do zmiennej `{{ .ToolCalls }}` - dzięki temu parser Ollamy potrafi wykryć prefix i wyciągnąć strukturyzowane wywołania funkcji. Szczegóły co dokładnie zmieniamy względem oryginalnego Modelfile'a - patrz `010-00. Bielik.Modelfile.md`.

## Wymagania

1. Zainstalowana Ollama: https://ollama.com/download
2. Pobrany model Bielika: `ollama pull SpeakLeash/bielik-11b-v3.0-instruct:bf16`
3. Zbudowany model `bielik-tools` z customowego Modelfile:
   ```bash
   ollama create bielik-tools -f "010-00. Bielik.Modelfile"
   ```
4. Działający serwis Ollama w tle (port `11434`).

> **Uwaga dot. DevContainera:** Notebook działa wewnątrz kontenera Dockera, więc `localhost` z perspektywy notebooka wskazuje na kontener, a nie hosta z uruchomioną Ollamą. W kodzie używamy `host.docker.internal`, aby z wnętrza kontenera dotrzeć do Ollamy działającej na hoście.

In [ ]:
import requests
import json

## Funkcja pobierająca współrzędne geograficzne dla danej nazwy

In [ ]:
def get_geolocation(location):
    """
    Pobiera współrzędne geograficzne oraz dane lokalizacyjne.

    Parametry:
    location (str): Nazwa lokalizacji, dla której chcemy uzyskać współrzędne geograficzne.

    Zwraca:
    dict: Dane lokalizacyjne w formacie JSON.
    """
    print(f"[tool_call] get_geolocation(location={location!r})")

    geocode_endpoint = "https://nominatim.openstreetmap.org/search"
    geocode_params = {"q": location, "format": "json"}
    headers = {"User-Agent": "Python script"}

    geocode_response = requests.get(geocode_endpoint, params=geocode_params, headers=headers)
    geocode_data = geocode_response.json()

    simplified_data = {
        "name": geocode_data[0]["display_name"],
        "latitude": geocode_data[0]["lat"],
        "longitude": geocode_data[0]["lon"]
    }
    return simplified_data

geolocation_data = get_geolocation("Poznań")
print(json.dumps(geolocation_data, indent=4, ensure_ascii=False))

## Funkcja pobierająca informacje o pogodzie dla podanych współrzędnych geograficznych

In [ ]:
def get_wind_direction(degrees):
    """Konwertuje kierunek wiatru ze stopni na nazwy kierunków świata."""
    directions = ['N', 'NNE', 'NE', 'ENE', 'E', 'ESE', 'SE', 'SSE',
                  'S', 'SSW', 'SW', 'WSW', 'W', 'WNW', 'NW', 'NNW']
    index = int((degrees + 11.25) // 22.5) % 16
    return directions[index]

def get_current_weather(latitude, longitude):
    """
    Pobiera aktualne dane pogodowe dla podanych współrzędnych geograficznych.

    Parametry:
    latitude (float): Szerokość geograficzna.
    longitude (float): Długość geograficzna.

    Zwraca:
    dict: Dane pogodowe w formacie JSON.
    """
    print(f"[tool_call] get_current_weather(latitude={latitude!r}, longitude={longitude!r})")

    weather_endpoint = "https://api.open-meteo.com/v1/forecast"
    weather_params = {
        "latitude": latitude,
        "longitude": longitude,
        "current_weather": True
    }

    weather_response = requests.get(weather_endpoint, params=weather_params)
    weather_data = weather_response.json()

    simplified_weather = {
        "temperature": f"{weather_data['current_weather']['temperature']} °C",
        "wind_speed": f"{weather_data['current_weather']['windspeed']} km/h",
        "wind_direction": get_wind_direction(weather_data['current_weather']['winddirection']),
        "is_day": "day" if weather_data['current_weather']['is_day'] else "night"
    }

    return simplified_weather

current_weather = get_current_weather(geolocation_data['latitude'], geolocation_data['longitude'])
print(json.dumps(current_weather, indent=4))

## Klient i schematy narzędzi

Klient OpenAI ustawiamy z `base_url` Ollamy i atrapą `api_key` (Ollama go ignoruje, ale klient OpenAI wymaga, żeby pole istniało). Schematy narzędzi piszemy ręcznie w formacie JSON Schema - dokładnie tak, jak oczekuje Chat Completions API.

In [ ]:
from openai import OpenAI

# Klient OpenAI wskazujący na lokalną Ollamę (kompatybilny endpoint /v1).
openai_client = OpenAI(
    base_url="http://host.docker.internal:11434/v1",
    api_key="ollama", # atrapa - Ollama ignoruje, ale klient OpenAI wymaga niepustego pola
)

# Definicje narzędzi w formacie wymaganym przez Chat Completions API.
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_geolocation",
            "description": "Pobiera współrzędne geograficzne (latitude, longitude) dla nazwy miejscowości.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "Nazwa miejscowości, np. 'Poznań'"
                    }
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Pobiera aktualne dane pogodowe dla podanych współrzędnych geograficznych.",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {"type": "number", "description": "Szerokość geograficzna"},
                    "longitude": {"type": "number", "description": "Długość geograficzna"}
                },
                "required": ["latitude", "longitude"]
            }
        }
    }
]

# Dispatcher - mapuje nazwę funkcji z tool_call na faktyczną funkcję Pythona.
tool_dispatch = {
    "get_geolocation": get_geolocation,
    "get_current_weather": get_current_weather,
}

## Pętla agenta

Klasyczna pętla function callingu: model dostaje historię wiadomości + listę narzędzi, decyduje czy wywołać funkcje, zwracamy mu wyniki, on dostaje kolejną szansę. Powtarzamy aż model przestanie prosić o tool calle albo wyczerpiemy `max_iters`.

In [ ]:
system_prompt = (
    "Jesteś pomocnym asystentem pogodowym. Jeśli do realizacji polecenia "
    "musisz ustalić współrzędne geograficzne jakiegoś miejsca, zawsze "
    "pobierz je za pomocą narzędzia get_geolocation - nigdy nie podawaj "
    "współrzędnych z własnej wiedzy. Nie każde polecenie wymaga "
    "ustalania współrzędnych."
)

def run_agent(user_question: str, max_iters: int = 6) -> str:
    """Manualna pętla function callingu na surowym SDK OpenAI."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_question},
    ]

    for iteration in range(max_iters):
        response = openai_client.chat.completions.create(
            model="bielik-tools",
            messages=messages,
            tools=tools_schema,
            temperature=0.3,
        )
        message = response.choices[0].message

        # Dopisanie odpowiedzi modelu do historii (wymagane przed dodaniem
        # wyników tool call - kolejność musi być zachowana).
        messages.append(message.model_dump(exclude_none=True))

        # Jeśli model nie poprosił o żadne narzędzie - kończymy.
        if not message.tool_calls:
            return message.content or "(brak odpowiedzi)"

        # Wykonanie każdego tool calla i dopisanie wyniku do historii
        # z rolą "tool" oraz powiązaniem przez tool_call_id.
        for tool_call in message.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)
            fn = tool_dispatch.get(fn_name)

            if fn is None:
                result = f"Błąd: nieznane narzędzie {fn_name}"
            else:
                result = fn(**fn_args)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result, ensure_ascii=False),
            })

    return "(przekroczono limit iteracji)"

answer = run_agent("Opisz jaka jest pogoda w Poznaniu. Czy powinienem wychodzić na spacer w stroju plażowym i okularach przeciwsłonecznych?")
print("\n=== Ostateczna odpowiedź agenta ===")
print(answer)